In [1]:
# 1. Install nnU-Net and SimpleITK
!pip install nnunetv2
!pip install --upgrade simpleitk

  Using cached argparse-1.4.0-py2.py3-none-any.whl.metadata (2.8 kB)
Using cached argparse-1.4.0-py2.py3-none-any.whl (23 kB)


# Restart and Countinue here

In [1]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/gdrive')

# Create directory for model storage
import os
GDRIVE_MODEL_DIR = '/content/gdrive/My Drive/nnUNet_Models'
os.makedirs(GDRIVE_MODEL_DIR, exist_ok=True)
print(f"✅ Google Drive mounted at: /content/gdrive")
print(f"✅ Model directory: {GDRIVE_MODEL_DIR}")

Mounted at /content/gdrive
✅ Google Drive mounted at: /content/gdrive
✅ Model directory: /content/gdrive/My Drive/nnUNet_Models


In [2]:
import os
import shutil
import json
import numpy as np
import SimpleITK as sitk
import torch
import gdown
import sys
import inspect
import nnunetv2
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, jaccard_score
from tqdm import tqdm
from PIL import Image

# 2. Download Data
file_id = '1Bv7zmjJVvE7uQbbjwRa-3qn-BI8h99KI'
url = f'https://drive.google.com/uc?id={file_id}'
output = 'data.zip'

if not os.path.exists(output):
    gdown.download(url, output, quiet=False)

if os.path.exists(output) and not os.path.exists("/content/data/data"):
    print("Unzipping...")
    !unzip -o -q data.zip -d /content/data
    print("✅ Data ready.")
else:
    print("✅ Data already extracted.")

# 3. Define Paths
SOURCE_IMG_DIR = "/content/data/data/images"
SOURCE_MASK_DIR = "/content/data/data/masks"

# 4. Set nnU-Net Environment Variables
os.environ['nnUNet_raw'] = "/content/nnUNet_raw"
os.environ['nnUNet_preprocessed'] = "/content/nnUNet_preprocessed"
os.environ['nnUNet_results'] = "/content/nnUNet_results"

for p in [os.environ['nnUNet_raw'], os.environ['nnUNet_preprocessed'], os.environ['nnUNet_results']]:
    os.makedirs(p, exist_ok=True)

Downloading...
From: https://drive.google.com/uc?id=1Bv7zmjJVvE7uQbbjwRa-3qn-BI8h99KI
To: /content/data.zip
100%|██████████| 18.0M/18.0M [00:00<00:00, 50.9MB/s]


Unzipping...
✅ Data ready.


In [3]:
import os
import inspect
import nnunetv2

# 1. Path to inject
nnunet_path = os.path.dirname(inspect.getfile(nnunetv2))
trainer_path = os.path.join(nnunet_path, "training", "nnUNetTrainer", "nnUNetTrainer_Custom.py")

# 2. custome Code (Using 'torch.device' explicitly)
Number_of_Epoch = 10
custom_code = f"""
import torch
from nnunetv2.training.nnUNetTrainer.nnUNetTrainer import nnUNetTrainer

class nnUNetTrainer_Custom(nnUNetTrainer):
    def __init__(self, plans: dict, configuration: str, fold: int, dataset_json: dict, device: torch.device = torch.device('cuda')):
        super().__init__(plans=plans, configuration=configuration, fold=fold, dataset_json=dataset_json, device=device)
        self.num_epochs = {Number_of_Epoch}
"""

# 3. Overwrite file
with open(trainer_path, "w") as f:
    f.write(custom_code)

print(f"✅ FIXED and overwritten trainer at: {trainer_path}")

✅ FIXED and overwritten trainer at: /usr/local/lib/python3.12/dist-packages/nnunetv2/training/nnUNetTrainer/nnUNetTrainer_Custom.py


In [4]:
def prepare_nnunet_data(seed, dataset_id=501):
    task_name = f"Dataset{dataset_id:03d}_Polyp"
    raw_base = os.path.join(os.environ['nnUNet_raw'], task_name)

    # Clean previous run to ensure fresh split
    if os.path.exists(raw_base): shutil.rmtree(raw_base)

    imagesTr = os.path.join(raw_base, "imagesTr")
    labelsTr = os.path.join(raw_base, "labelsTr")
    imagesTs = os.path.join(raw_base, "imagesTs")
    labelsTs = os.path.join(raw_base, "labelsTs")

    for p in [imagesTr, labelsTr, imagesTs, labelsTs]:
        os.makedirs(p, exist_ok=True)

    # Split Data
    all_files = sorted([f for f in os.listdir(SOURCE_IMG_DIR) if f.lower().endswith(('.jpg', '.png'))])
    train_files, test_files = train_test_split(all_files, test_size=0.2, random_state=seed)

    def convert_to_nifti(files, dest_img_dir, dest_lbl_dir, is_train=True):
        processed_ids = []
        for f in tqdm(files, desc=f"Converting {'Train' if is_train else 'Test'}"):
            img_path = os.path.join(SOURCE_IMG_DIR, f)
            base_name = os.path.splitext(f)[0]

            # Find Mask
            mask_path = None
            for ext in ['.png', '.jpg']:
                try_path = os.path.join(SOURCE_MASK_DIR, base_name + ext)
                if os.path.exists(try_path): mask_path = try_path; break

            if not mask_path: continue

            # --- CRITICAL FIX: Convert to Grayscale (L) ---
            img_pil = Image.open(img_path).convert("L")
            mask_pil = Image.open(mask_path).convert("L")

            img_arr = np.array(img_pil)
            mask_arr = np.array(mask_pil)
            mask_arr = (mask_arr > 0).astype(np.uint8) # Threshold to Binary

            # Create SimpleITK Images
            img_obj = sitk.GetImageFromArray(img_arr)
            mask_obj = sitk.GetImageFromArray(mask_arr)

            # Copy Metadata (Required for nnU-Net)
            img_obj.SetSpacing((1.0, 1.0))
            mask_obj.SetSpacing((1.0, 1.0))
            img_obj.SetOrigin((0.0, 0.0))
            mask_obj.SetOrigin((0.0, 0.0))

            # Save
            sitk.WriteImage(img_obj, os.path.join(dest_img_dir, f"{base_name}_0000.nii.gz"))
            if is_train:
                sitk.WriteImage(mask_obj, os.path.join(dest_lbl_dir, f"{base_name}.nii.gz"))
            else:
                # Save GT to labelsTs for our evaluation script
                sitk.WriteImage(mask_obj, os.path.join(dest_lbl_dir, f"{base_name}.nii.gz"))

            processed_ids.append(base_name)
        return processed_ids

    train_ids = convert_to_nifti(train_files, imagesTr, labelsTr, is_train=True)
    test_ids = convert_to_nifti(test_files, imagesTs, labelsTs, is_train=False)

    # Create dataset.json with channel "Gray"
    json_dict = {
        "channel_names": {"0": "Gray"},
        "labels": {"background": 0, "polyp": 1},
        "numTraining": len(train_ids),
        "file_ending": ".nii.gz"
    }
    with open(os.path.join(raw_base, "dataset.json"), 'w') as f:
        json.dump(json_dict, f)

    return task_name, test_ids

In [5]:
def evaluate_nnunet_predictions(task_name, test_ids):
    # Predictions are saved here by the 'predict' command
    pred_dir = os.path.join(os.environ['nnUNet_results'], task_name, "inference_output")
    gt_dir = os.path.join(os.environ['nnUNet_raw'], task_name, "labelsTs")

    scores = {"dice": [], "iou": [], "prec": [], "rec": [], "acc": []}

    print("Evaluating Predictions...")
    found_count = 0
    for case_id in test_ids:
        pred_path = os.path.join(pred_dir, f"{case_id}.nii.gz")
        gt_path = os.path.join(gt_dir, f"{case_id}.nii.gz")

        if not os.path.exists(pred_path): continue
        found_count += 1

        pred = sitk.GetArrayFromImage(sitk.ReadImage(pred_path)).flatten()
        gt = sitk.GetArrayFromImage(sitk.ReadImage(gt_path)).flatten()

        pred = (pred > 0).astype(np.uint8)
        gt = (gt > 0).astype(np.uint8)

        scores["dice"].append(f1_score(gt, pred, zero_division=1))
        scores["iou"].append(jaccard_score(gt, pred, zero_division=1))
        scores["prec"].append(precision_score(gt, pred, zero_division=1))
        scores["rec"].append(recall_score(gt, pred, zero_division=1))
        scores["acc"].append(accuracy_score(gt, pred))

    if found_count == 0:
        return {k: 0.0 for k in scores}

    return {k: np.mean(v) for k, v in scores.items()}

In [6]:
# SEEDS = [42, 0, 123, 7, 2026]
SEEDS = [42]
DATASET_ID = 501
all_results = {"dice": [], "iou": [], "prec": [], "rec": [], "acc": []}

for seed in SEEDS:
    print(f"\n{'='*40}")
    print(f"🚀 STARTING RUN WITH SEED: {seed}")
    print(f"{'='*40}")

    # 1. Prepare Data
    task_name, test_ids = prepare_nnunet_data(seed, dataset_id=DATASET_ID)

    # 2. Plan & Preprocess
    print("Step: Planning & Preprocessing...")
    !nnUNetv2_plan_and_preprocess -d {DATASET_ID} --verify_dataset_integrity -c 2d

    # 3. Train (Using injected Custom Trainer)
    print(f"Step: Training ({Number_of_Epoch} Epochs)...")
    !nnUNetv2_train {DATASET_ID} 2d 0 -tr nnUNetTrainer_Custom -p nnUNetPlans

    # 4. Predict (Explicit Inference Step)
    print("Step: Inference on Test Set...")
    input_folder = os.path.join(os.environ['nnUNet_raw'], task_name, "imagesTs")
    output_folder = os.path.join(os.environ['nnUNet_results'], task_name, "inference_output")

    !nnUNetv2_predict -i {input_folder} -o {output_folder} -d {DATASET_ID} -c 2d -f 0 -tr nnUNetTrainer_Custom -p nnUNetPlans

    # 5. Evaluate
    metrics = evaluate_nnunet_predictions(task_name, test_ids)

    for k, v in metrics.items():
        all_results[k].append(v)

    print(f"Run {seed} Results -> Dice: {metrics['dice']:.4f} | IoU: {metrics['iou']:.4f}")

# --- Final Report ---
print("\n" + "="*50)
print("✅ FINAL MULTI-RUN REPORT (nnU-Net)")
print("="*50)
print(f"{'Metric':<10} | {'Mean':<10} | {'Std Dev':<10}")
print("-" * 35)

for k, v in all_results.items():
    print(f"{k.upper():<10} | {np.mean(v):.4f}     | {np.std(v):.4f}")


🚀 STARTING RUN WITH SEED: 42


Converting Test: 100%|██████████| 87/87 [00:01<00:00, 78.13it/s]

Step: Planning & Preprocessing...
Step: Training (10 Epochs)...
Step: Inference on Test Set...
Evaluating Predictions...
Run 42 Results -> Dice: 0.0000 | IoU: 0.0000

✅ FINAL MULTI-RUN REPORT (nnU-Net)
Metric     | Mean       | Std Dev   
-----------------------------------
DICE       | 0.0000     | 0.0000
IOU        | 0.0000     | 0.0000
PREC       | 0.0000     | 0.0000
REC        | 0.0000     | 0.0000
ACC        | 0.0000     | 0.0000


In [14]:
# Save trained model to Google Drive
import shutil
import os
from datetime import datetime

DATASET_ID = 501
MODEL_NAME = f"nnUNet_polyp_trained_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

# Source model directory from training
source_model_dir = os.path.join(
    os.environ['nnUNet_results'],
    f"Dataset{DATASET_ID:03d}_Polyp",
    "nnUNetTrainer_Custom__nnUNetPlans__2d"
)

# Destination in Google Drive
dest_model_dir = os.path.join(GDRIVE_MODEL_DIR, MODEL_NAME)

print(f"📦 Saving model from: {source_model_dir}")
print(f"📦 Saving model to:   {dest_model_dir}")

if os.path.exists(source_model_dir):
    shutil.copytree(source_model_dir, dest_model_dir, dirs_exist_ok=True)
    print(f"✅ Model successfully saved to Google Drive!")
    print(f"   Model name: {MODEL_NAME}")
else:
    print(f"❌ Error: Model directory not found at {source_model_dir}")

# Save model path for reference
with open(os.path.join(GDRIVE_MODEL_DIR, 'latest_model.txt'), 'w') as f:
    f.write(MODEL_NAME)

📦 Saving model from: /content/nnUNet_results/Dataset501_Polyp/nnUNetTrainer_Custom__nnUNetPlans__2d
📦 Saving model to:   /content/gdrive/My Drive/nnUNet_Models/nnUNet_polyp_trained_20260112_132106
✅ Model successfully saved to Google Drive!
   Model name: nnUNet_polyp_trained_20260112_132106


# Stop Training here, comment train commands, skip previous block and rerun with T4


In [7]:
# Load model from Google Drive for Inference
from google.colab import drive
import os
import shutil

print("🔄 Loading trained model from Google Drive...")

# Mount Google Drive if not already mounted
try:
    if not os.path.exists('/content/gdrive/My Drive'):
        drive.mount('/content/gdrive')
except:
    pass

GDRIVE_MODEL_DIR = '/content/gdrive/My Drive/nnUNet_Models'

# Read latest model name
latest_model_file = os.path.join(GDRIVE_MODEL_DIR, 'latest_model.txt')
if os.path.exists(latest_model_file):
    with open(latest_model_file, 'r') as f:
        MODEL_NAME = f.read().strip()
    print(f"✅ Found latest model: {MODEL_NAME}")
else:
    # If no latest file, list available models
    models = [d for d in os.listdir(GDRIVE_MODEL_DIR) if d.startswith('nnUNet_polyp_trained_')]
    if models:
        MODEL_NAME = sorted(models)[-1]  # Get most recent
        print(f"✅ Using latest model: {MODEL_NAME}")
    else:
        raise FileNotFoundError("No trained models found in Google Drive!")

# Copy model from Google Drive to local nnUNet_results
source_model_gdrive = os.path.join(GDRIVE_MODEL_DIR, MODEL_NAME)
DATASET_ID = 501
dest_model_local = os.path.join(
    os.environ['nnUNet_results'],
    f"Dataset{DATASET_ID:03d}_Polyp",
    "nnUNetTrainer_Custom__nnUNetPlans__2d"
)

print(f"📥 Copying model from Google Drive...")
print(f"   From: {source_model_gdrive}")
print(f"   To:   {dest_model_local}")

os.makedirs(os.path.dirname(dest_model_local), exist_ok=True)
if os.path.exists(dest_model_local):
    shutil.rmtree(dest_model_local)
shutil.copytree(source_model_gdrive, dest_model_local)

print(f"✅ Model loaded successfully from Google Drive!")

🔄 Loading trained model from Google Drive...
✅ Found latest model: nnUNet_polyp_trained_20260112_132106
📥 Copying model from Google Drive...
   From: /content/gdrive/My Drive/nnUNet_Models/nnUNet_polyp_trained_20260112_132106
   To:   /content/nnUNet_results/Dataset501_Polyp/nnUNetTrainer_Custom__nnUNetPlans__2d
✅ Model loaded successfully from Google Drive!


In [8]:
import os
import time
import numpy as np
import torch
import SimpleITK as sitk
from nnunetv2.inference.predict_from_raw_data import nnUNetPredictor

print("⏱️ Benchmarking Inference Speed (nnU-Net) - 2D config (Z=1 container)...")

# --- Configuration ---
DATASET_ID = 501
ITERATIONS = 50
WARMUP = 10
USE_FOLDS = (0,)                 # single fold for benchmarking
CHECKPOINT_NAME = "checkpoint_final.pth"
# ---------------------

# 1) Initialize Predictor
predictor = nnUNetPredictor(
    tile_step_size=0.5,
    use_gaussian=True,
    use_mirroring=False,
    perform_everything_on_device=True,
    device=torch.device("cuda"),
    verbose=False,
    verbose_preprocessing=False,
    allow_tqdm=False,
)

checkpoint_path = os.path.join(
    os.environ["nnUNet_results"],
    f"Dataset{DATASET_ID:03d}_Polyp",
    "nnUNetTrainer_Custom__nnUNetPlans__2d",
)
if not os.path.isdir(checkpoint_path):
    raise FileNotFoundError(f"Checkpoint folder not found: {checkpoint_path}")

predictor.initialize_from_trained_model_folder(
    checkpoint_path,
    use_folds=USE_FOLDS,
    checkpoint_name=CHECKPOINT_NAME,
)

# 2) Pick one test file
test_folder = os.path.join(os.environ["nnUNet_raw"], f"Dataset{DATASET_ID:03d}_Polyp", "imagesTs")
if not os.path.isdir(test_folder):
    raise FileNotFoundError(f"Test folder not found: {test_folder}")

test_files = sorted([f for f in os.listdir(test_folder) if f.endswith("_0000.nii.gz")])
if not test_files:
    raise FileNotFoundError(f"No *_0000.nii.gz found in: {test_folder}")

img_path = os.path.join(test_folder, test_files[0])
print(f"   Using file: {test_files[0]}")

sitk_img = sitk.ReadImage(img_path)
arr = sitk.GetArrayFromImage(sitk_img).astype(np.float32)

print(f"   SITK size (x,y,z?): {sitk_img.GetSize()}")
print(f"   Loaded numpy shape: {arr.shape}")
print(f"   transpose_forward:  {predictor.plans_manager.transpose_forward}")

# 3) IMPORTANT FIX
# Your plans_manager.transpose_forward is [0, 1, 2] -> nnU-Net expects 3 spatial dims.
# Even for a '2D' config, nnU-Net v2 can internally represent data as (C, Z=1, Y, X).
# So we MUST feed (C, 1, Y, X), not (C, Y, X).

if arr.ndim == 2:
    arr = arr[None, ...]          # (Z=1, Y, X)
elif arr.ndim == 3:
    # If you ever get true 3D here, keep it; for pure 2D use a single slice.
    # Uncomment to force a single slice benchmark:
    # arr = arr[arr.shape[0] // 2 : arr.shape[0] // 2 + 1]
    pass
else:
    raise RuntimeError(f"Unsupported ndim={arr.ndim} with shape={arr.shape}")

# now arr is (Z, Y, X)
input_data = arr[None, ...]       # (C=1, Z, Y, X)
print(f"   Final input_data shape (C,Z,Y,X): {input_data.shape}")

# 4) Properties: spacing must match spatial dims (Z, Y, X) in numpy order.
sp = tuple(float(x) for x in sitk_img.GetSpacing())
# sitk spacing order is (sx, sy[, sz])
if len(sp) == 2:
    sx, sy = sp
    sz = 1.0
elif len(sp) == 3:
    sx, sy, sz = sp
else:
    raise RuntimeError(f"Unexpected spacing: {sp}")

# nnU-Net expects spacing in numpy axis order (Z, Y, X)
props = {"spacing": (sz, sy, sx), "sitk_stuff": {}}
print(f"   Using spacing (sz,sy,sx): {props['spacing']}")

# 5) Sanity: CUDA
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is NOT available. You are benchmarking on CPU.")
print(f"   CUDA device: {torch.cuda.get_device_name(0)}")

# 6) Warmup
print("   Warming up...")
for _ in range(WARMUP):
    _ = predictor.predict_single_npy_array(input_data, props, None, None, False)

torch.cuda.synchronize()

# 7) Measure end-to-end nnU-Net prediction time
print(f"   Measuring ({ITERATIONS} runs)...")
timings_ms = []
for _ in range(ITERATIONS):
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    _ = predictor.predict_single_npy_array(input_data, props, None, None, False)
    torch.cuda.synchronize()
    t1 = time.perf_counter()
    timings_ms.append((t1 - t0) * 1000.0)

avg_ms = float(np.mean(timings_ms))
p50 = float(np.percentile(timings_ms, 50))
p90 = float(np.percentile(timings_ms, 90))
p99 = float(np.percentile(timings_ms, 99))
fps = 1000.0 / avg_ms

print("\n" + "=" * 30)
print("🚀 FINAL SPEED RESULTS (nnU-Net end-to-end)")
print("=" * 30)
print(f"Avg Time: {avg_ms:.2f} ms")
print(f"P50:      {p50:.2f} ms")
print(f"P90:      {p90:.2f} ms")
print(f"P99:      {p99:.2f} ms")
print(f"FPS:      {fps:.2f} frames/sec")
print("=" * 30)

import time
import numpy as np
import torch
import torch.nn.functional as F


###############################################################################
# --- Forward-only benchmark (pure network forward pass) ---                  #
# Assumes you already have:                                                   #
#   - predictor                                                               #
#   - input_data shaped (C, Z, Y, X) with Z=1                                 #

print("\n⚡ Forward-only benchmark (pure network forward pass, padded)...")
net = predictor.network
net.eval()

# Build 4D tensor for a 2D UNet: (B, C, H, W)
# input_data: (C, Z, Y, X) -> take Z=0 -> (C, Y, X) then add batch
x4 = torch.from_numpy(input_data[:, 0, ...]).unsqueeze(0).to(predictor.device)  # (1, C, H, W)

# nnU-Net's full predictor pads/crops internally so shapes are compatible with all skip connections.
# When you call net(x) directly, you must ensure H and W are divisible by the total downsampling factor.

# Infer downsampling depth; for most nnU-Net UNets, each stage downsamples by 2.
# PlainConvEncoder has `stages`; first stage typically has no downsample.
try:
    num_stages = len(net.encoder.stages)
    num_pools = max(0, num_stages - 1)
except Exception:
    # Safe fallback (common for nnU-Net 2D)
    num_pools = 5

divisor = 2 ** num_pools

B, C, H, W = x4.shape
pad_h = (divisor - (H % divisor)) % divisor
pad_w = (divisor - (W % divisor)) % divisor

# Pad only on the right/bottom (fast, and consistent enough for timing)
# F.pad format for 2D: (left, right, top, bottom)
x4p = F.pad(x4, (0, pad_w, 0, pad_h))

print(f"   Input HxW: {H}x{W} -> padded to {x4p.shape[-2]}x{x4p.shape[-1]} (divisor={divisor})")

# Warmup
with torch.no_grad():
    for _ in range(20):
        _ = net(x4p)
torch.cuda.synchronize()

ITER2 = 200
tim2 = []
with torch.no_grad():
    for _ in range(ITER2):
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        _ = net(x4p)
        torch.cuda.synchronize()
        t1 = time.perf_counter()
        tim2.append((t1 - t0) * 1000.0)

avg2 = float(np.mean(tim2))
p50 = float(np.percentile(tim2, 50))
p90 = float(np.percentile(tim2, 90))

print(f"Forward-only Avg: {avg2:.4f} ms | P50: {p50:.4f} ms | P90: {p90:.4f} ms | FPS: {1000.0/avg2:.2f}")

# Note:
# - This is NOT the same as nnU-Net end-to-end inference. It times only the UNet forward.
# - Padding changes the compute slightly; for apples-to-apples, always pad the same way.




⏱️ Benchmarking Inference Speed (nnU-Net) - 2D config (Z=1 container)...
   Using file: group10_1402-09-2810393516-5643_0000.nii.gz
   SITK size (x,y,z?): (672, 480)
   Loaded numpy shape: (480, 672)
   transpose_forward:  [0, 1, 2]
   Final input_data shape (C,Z,Y,X): (1, 1, 480, 672)
   Using spacing (sz,sy,sx): (1.0, 1.0, 1.0)
   CUDA device: Tesla T4
   Warming up...
   Measuring (50 runs)...

🚀 FINAL SPEED RESULTS (nnU-Net end-to-end)
Avg Time: 188.48 ms
P50:      181.84 ms
P90:      210.47 ms
P99:      222.72 ms
FPS:      5.31 frames/sec

⚡ Forward-only benchmark (pure network forward pass, padded)...
   Input HxW: 480x672 -> padded to 512x768 (divisor=128)
Forward-only Avg: 48.8524 ms | P50: 48.6904 ms | P90: 49.6913 ms | FPS: 20.47
